# 3E Kalite Kontrol PoC — YOLO11n Kusur Dedektörü (Colab, ayrı oturum)

Bu notebook sıfırdan bir Colab oturumunda (T4 GPU) çalışır: MVTec AD veri kümesinden 3 ana uygulama kategorisi ve 7 ek kategoriyi indirir, gerçek kusur maskelerinden YOLO sınır kutusu (bounding box) etiketleri üretir, offline veri artırımı ile >= 1000 eğitim örneğine ulaşır, YOLO11n modelini eğitir, doğrular ve ONNX formatına aktarır.

- **Uygulama Kategorileri**: `metal_nut` → Optik lens grubu, `transistor` → Termal kamera modülü, `cable` → Gözetleme ünitesi
- **Ek Kategoriler (Eğitim havuzunu büyütmek için)**: `screw`, `capsule`, `pill`, `bottle`, `hazelnut`, `wood`, `tile`
- **Sınıflar**: `0 scratch` (Yüzey çiziği/kırık), `1 coating` (Kaplama/baskı/leke kusuru), `2 connector_assembly` (Konektör/montaj/bükülme kusuru)
- **Veri Yaklaşımı**: Sentetik copy-paste yerine yalnız GERÇEK MVTec AD kusur görselleri kullanılır. Eğitim seti geometrik (çevirme, rotasyon) ve fotometrik (parlaklık, kontrast, hafif gürültü) offline veri artırımıyla ~1000 train hedefine genişletilir; val (doğrulama) seti tamamen gerçek ve artırımsız kalır.
- **Lisans**: MVTec AD, CC BY-NC-SA 4.0 (ticari olmayan kullanım)

> Not: MVTec'te etiketli (maskeli) kusur görselleri test bölümündedir; eğitim ve doğrulama bunlarla yapılır. Metrikler ayrılan %20 doğrulama setinde ve ayrıca uygulama kategorileri özelinde (`mAP50_app`) ölçülür.

**Runtime → Change runtime type → T4 GPU** seçip hücreleri sırayla çalıştırın.

## 1. Kurulum ve GPU kontrolü

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics huggingface_hub

## 2. Veri indirme (Hugging Face aynası — 3 uygulama + 7 ek kategori)

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

CATEGORIES = ['metal_nut', 'transistor', 'cable']
EXTRA_CATEGORIES = ['screw', 'capsule', 'pill', 'bottle', 'hazelnut', 'wood', 'tile']
REPO = 'XimiaoZhang/MVTec-FS'  # mirror with test/<defect> images and ground_truth masks
# If the download asks for authentication: from huggingface_hub import login; login()
local = snapshot_download(
    REPO, repo_type='dataset', local_dir='mvtec_dl',
    allow_patterns=[f'mvtec/images/{c}/*' for c in CATEGORIES + EXTRA_CATEGORIES],
)
MVTEC_ROOT = Path(local) / 'mvtec' / 'images'
for c in CATEGORIES + EXTRA_CATEGORIES:
    n_def = len(list((MVTEC_ROOT / c / 'test').glob('*/*.png')))
    n_gt = len(list((MVTEC_ROOT / c / 'ground_truth').glob('*/*.png')))
    print(f'{c}: test images={n_def}, masks={n_gt}')

## 3. Maskelerden YOLO etiketleri ve train/val bölmesi

In [ ]:
import cv2, random, shutil, yaml
from pathlib import Path

CLASSES = ['scratch', 'coating', 'connector_assembly']
MAP = {
    'metal_nut': {'scratch': 0, 'color': 1, 'bent': 2, 'flip': 2},
    'transistor': {'damaged_case': 0, 'bent_lead': 2, 'cut_lead': 2, 'misplaced': 2},
    'cable': {'cut_inner_insulation': 1, 'cut_outer_insulation': 1, 'poke_insulation': 1,
              'bent_wire': 2, 'cable_swap': 2, 'missing_cable': 2, 'missing_wire': 2},
    'screw': {'scratch_head': 0, 'scratch_neck': 0, 'manipulated_front': 2, 'thread_side': 2, 'thread_top': 2},
    'capsule': {'scratch': 0, 'crack': 0, 'poke': 0, 'faulty_imprint': 1},
    'pill': {'scratch': 0, 'crack': 0, 'color': 1, 'contamination': 1, 'faulty_imprint': 1},
    'bottle': {'broken_large': 0, 'broken_small': 0, 'contamination': 1},
    'hazelnut': {'crack': 0, 'cut': 0, 'print': 1},
    'wood': {'scratch': 0, 'color': 1, 'liquid': 1},
    'tile': {'crack': 0, 'oil': 1, 'gray_stroke': 1},
}
MIN_AREA = 30
N_NEG = 15
random.seed(0)

DS = Path('yolo_ds')
if DS.exists():
    shutil.rmtree(DS)
for s in ('train', 'val'):
    (DS / 'images' / s).mkdir(parents=True)
    (DS / 'labels' / s).mkdir(parents=True)

def boxes_from_mask(mask_path, w, h):
    m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if m is None:
        return []
    m = (m > 0).astype('uint8') * 255  # mirror masks may store label ids (1..k) instead of 255
    n, _, stats, _ = cv2.connectedComponentsWithStats(m)
    out = []
    for i in range(1, n):
        x, y, bw, bh, area = stats[i]
        if area < MIN_AREA:
            continue
        out.append(((x + bw / 2) / w, (y + bh / 2) / h, bw / w, bh / h))
    return out

stats_count = {}
val_app_images = []
all_cats = CATEGORIES + EXTRA_CATEGORIES

for cat in all_cats:
    cat_dir = MVTEC_ROOT / cat
    if not cat_dir.exists():
        continue
    items = []  # (img_path, [label lines])
    test_dir = cat_dir / 'test'
    if not test_dir.exists():
        continue
    for d in sorted(test_dir.iterdir()):
        if d.name == 'good':
            continue
        if cat not in MAP or d.name not in MAP[cat]:
            print(f'skip {cat}/{d.name} (unmapped)')
            continue
        cls = MAP[cat][d.name]
        for img in sorted(d.glob('*.png')):
            im = cv2.imread(str(img))
            if im is None:
                continue
            h, w = im.shape[:2]
            mask = MVTEC_ROOT / cat / 'ground_truth' / d.name / f'{img.stem}_mask.png'
            bxs = boxes_from_mask(mask, w, h)
            if not bxs:
                continue
            items.append((img, [f'{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}' for cx, cy, bw, bh in bxs]))
            stats_count[CLASSES[cls]] = stats_count.get(CLASSES[cls], 0) + len(bxs)
    good_dir = test_dir / 'good'
    goods = sorted(good_dir.glob('*.png')) if good_dir.exists() else []
    random.shuffle(goods)
    items += [(g, []) for g in goods[:N_NEG]]  # negatives with empty labels
    random.shuffle(items)
    n_val = max(1, int(0.2 * len(items))) if items else 0
    for i, (img, lines) in enumerate(items):
        split = 'val' if i < n_val else 'train'
        name = f'{cat}_{img.parent.name}_{img.stem}'
        dest_img = DS / 'images' / split / f'{name}.png'
        shutil.copy(img, dest_img)
        (DS / 'labels' / split / f'{name}.txt').write_text('\n'.join(lines))
        if split == 'val' and cat in CATEGORIES:
            val_app_images.append(str(dest_img.resolve()))
    print(f'{cat}: {len(items)} images ({n_val} val)')

(DS / 'val_app.txt').write_text('\n'.join(val_app_images) + ('\n' if val_app_images else ''))

(DS / 'data.yaml').write_text(yaml.safe_dump({
    'path': str(DS.resolve()), 'train': 'images/train', 'val': 'images/val',
    'names': {i: n for i, n in enumerate(CLASSES)},
}))
print(f'val_app.txt written with {len(val_app_images)} images from app categories {CATEGORIES}')
print('boxes per class:', stats_count)

## 3b. Gerçek görsellerle offline augmentation

Sentetik copy-paste yerine yalnızca gerçek MVTec AD kusur görselleri kullanılarak geometrik (hflip, vflip, rot90) ve fotometrik (parlaklık, kontrast, hafif gürültü) offline veri artırımı yapılır.
İşlem yalnızca eğitim (train) seti üzerinde uygulanır; doğrulama (val) seti tamamen gerçek ve artırımsız tutulur.
YOLO sınır kutusu etiketleri geometrik dönüşümlere göre tam matematiksel uyumla güncellenir.

In [ ]:
import cv2, random, numpy as np
from pathlib import Path

# Öz-test: Kutu koordinat dönüşümlerinin numpy matris dönüşümleriyle tam uyumunu doğrula
def _self_test_box_transforms():
    H, W = 100, 200
    x1, x2, y1, y2 = 20, 60, 10, 50
    mask = np.zeros((H, W), dtype=np.uint8)
    mask[y1:y2, x1:x2] = 255
    cx, cy = (x1 + x2) / (2.0 * W), (y1 + y2) / (2.0 * H)
    bw, bh = (x2 - x1) / float(W), (y2 - y1) / float(H)

    # 1. hflip
    t_img = np.fliplr(mask)
    t_box = (1.0 - cx, cy, bw, bh)
    ys, xs = np.where(t_img > 0)
    assert abs(t_box[0] - (xs.min() + xs.max() + 1) / (2.0 * t_img.shape[1])) < 1e-4
    assert abs(t_box[1] - (ys.min() + ys.max() + 1) / (2.0 * t_img.shape[0])) < 1e-4
    assert abs(t_box[2] - (xs.max() - xs.min() + 1) / float(t_img.shape[1])) < 1e-4
    assert abs(t_box[3] - (ys.max() - ys.min() + 1) / float(t_img.shape[0])) < 1e-4

    # 2. vflip
    t_img = np.flipud(mask)
    t_box = (cx, 1.0 - cy, bw, bh)
    ys, xs = np.where(t_img > 0)
    assert abs(t_box[0] - (xs.min() + xs.max() + 1) / (2.0 * t_img.shape[1])) < 1e-4
    assert abs(t_box[1] - (ys.min() + ys.max() + 1) / (2.0 * t_img.shape[0])) < 1e-4
    assert abs(t_box[2] - (xs.max() - xs.min() + 1) / float(t_img.shape[1])) < 1e-4
    assert abs(t_box[3] - (ys.max() - ys.min() + 1) / float(t_img.shape[0])) < 1e-4

    # 3. rot90 k=1, 2, 3
    for k in (1, 2, 3):
        t_img = np.rot90(mask, k)
        if k == 1:
            t_box = (cy, 1.0 - cx, bh, bw)
        elif k == 2:
            t_box = (1.0 - cx, 1.0 - cy, bw, bh)
        elif k == 3:
            t_box = (1.0 - cy, cx, bh, bw)
        ys, xs = np.where(t_img > 0)
        assert abs(t_box[0] - (xs.min() + xs.max() + 1) / (2.0 * t_img.shape[1])) < 1e-4
        assert abs(t_box[1] - (ys.min() + ys.max() + 1) / (2.0 * t_img.shape[0])) < 1e-4
        assert abs(t_box[2] - (xs.max() - xs.min() + 1) / float(t_img.shape[1])) < 1e-4
        assert abs(t_box[3] - (ys.max() - ys.min() + 1) / float(t_img.shape[0])) < 1e-4

_self_test_box_transforms()
print('Self-test passed: Box transformations match numpy image operations.')

random.seed(0)
np.random.seed(0)

TARGET_TRAIN = 1000
train_img_dir = DS / 'images' / 'train'
train_lbl_dir = DS / 'labels' / 'train'

orig_train_images = sorted(train_img_dir.glob('*.png'))
orig_train_count = len(orig_train_images)

# Yalnızca dolu etiketli (kusurlu) eğitim görsellerini bul
non_empty_items = []
for img_path in orig_train_images:
    lbl_path = train_lbl_dir / f'{img_path.stem}.txt'
    if not lbl_path.exists():
        continue
    content = lbl_path.read_text().strip()
    if not content:
        continue
    boxes = []
    for line in content.splitlines():
        parts = line.strip().split()
        if len(parts) == 5:
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            boxes.append((cls_id, cx, cy, bw, bh))
    if boxes:
        non_empty_items.append((img_path, lbl_path, boxes))

n_def = len(non_empty_items)
if n_def > 0 and orig_train_count < TARGET_TRAIN:
    needed = TARGET_TRAIN - orig_train_count
    AUG_PER_IMAGE = min(3, max(1, (needed + n_def - 1) // n_def))
else:
    AUG_PER_IMAGE = 1

print(f'Original train count: {orig_train_count}, Non-empty defect train images: {n_def}')
print(f'AUG_PER_IMAGE: {AUG_PER_IMAGE} (target: {TARGET_TRAIN})')

added_count = 0
for img_path, lbl_path, boxes in non_empty_items:
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    stem = img_path.stem

    for k in range(1, AUG_PER_IMAGE + 1):
        aug_name = f'aug_{k}_{stem}'
        geom = random.choice(['hflip', 'vflip', 'rot90'])

        aug_img = img.copy()
        aug_boxes = []
        if geom == 'hflip':
            aug_img = np.ascontiguousarray(np.fliplr(aug_img))
            for cls_id, cx, cy, bw, bh in boxes:
                aug_boxes.append((cls_id, 1.0 - cx, cy, bw, bh))
        elif geom == 'vflip':
            aug_img = np.ascontiguousarray(np.flipud(aug_img))
            for cls_id, cx, cy, bw, bh in boxes:
                aug_boxes.append((cls_id, cx, 1.0 - cy, bw, bh))
        elif geom == 'rot90':
            rot_k = random.choice([1, 2, 3])
            aug_img = np.ascontiguousarray(np.rot90(aug_img, rot_k))
            for cls_id, cx, cy, bw, bh in boxes:
                if rot_k == 1:
                    ncx, ncy, nbw, nbh = cy, 1.0 - cx, bh, bw
                elif rot_k == 2:
                    ncx, ncy, nbw, nbh = 1.0 - cx, 1.0 - cy, bw, bh
                elif rot_k == 3:
                    ncx, ncy, nbw, nbh = 1.0 - cy, cx, bh, bw
                aug_boxes.append((cls_id, ncx, ncy, nbw, nbh))

        # Fotometrik jitter: parlaklık, kontrast ve hafif gauss gürültüsü
        alpha = random.uniform(0.85, 1.15)
        beta = random.uniform(-15.0, 15.0)
        noise = np.random.normal(0, 5.0, aug_img.shape)
        aug_img = np.clip(aug_img.astype(np.float32) * alpha + beta + noise, 0, 255).astype(np.uint8)

        # Görseli kaydet
        cv2.imwrite(str(train_img_dir / f'{aug_name}.png'), aug_img)

        # Dönüştürülmüş etiketleri kaydet ([0, 1] aralığına kırparak)
        lines = []
        for cls_id, cx, cy, bw, bh in aug_boxes:
            cx = min(max(cx, 0.0), 1.0)
            cy = min(max(cy, 0.0), 1.0)
            bw = min(max(bw, 0.0), 1.0)
            bh = min(max(bh, 0.0), 1.0)
            lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        (train_lbl_dir / f'{aug_name}.txt').write_text('\n'.join(lines) + '\n')
        added_count += 1

final_train_count = len(list(train_img_dir.glob('*.png')))
print(f'Train images: original={orig_train_count}, added={added_count}, final={final_train_count}')

## 4. YOLO11n eğitimi (T4'te ~5-10 dk)

In [ ]:
import time
from ultralytics import YOLO

t0 = time.time()
model = YOLO('yolo11n.pt')
model.train(data=str(DS / 'data.yaml'), imgsz=640, epochs=80, patience=20, batch=16, seed=0,
            fliplr=0.5, flipud=0.5, degrees=10, project='runs', name='mvtec_y11n', exist_ok=True,
            plots=True, verbose=False)
print(f'training time: {(time.time() - t0) / 60:.1f} min')

## 4b. (Opsiyonel) Eğitime devam

runs/mvtec_y11n/results.png içindeki doğrulama mAP50 eğrisi son epoch'ta hâlâ yükseliyorsa bu bölümü çalıştırın.
best.pt ağırlıklarından başlayarak daha düşük öğrenme oranıyla (lr0=0.002) 60 epoch daha eğitime devam eder.
Yalnızca doğrulama mAP50 skoru gelişirse runs/mvtec_y11n/weights/best.pt güncellenir; ardından 5-7 bölümlerini tekrar çalıştırın.

In [ ]:
from ultralytics import YOLO
import shutil

CONTINUE_EPOCHS = 60
base_map = YOLO('runs/mvtec_y11n/weights/best.pt').val(data=str(DS / 'data.yaml'), imgsz=640, verbose=False).box.map50
m2 = YOLO('runs/mvtec_y11n/weights/best.pt')
m2.train(data=str(DS / 'data.yaml'), imgsz=640, epochs=CONTINUE_EPOCHS, patience=20, batch=16, seed=0, lr0=0.002, warmup_epochs=0, fliplr=0.5, flipud=0.5, degrees=10, project='runs', name='mvtec_y11n_cont', exist_ok=True, plots=True, verbose=False)
new_map = YOLO('runs/mvtec_y11n_cont/weights/best.pt').val(data=str(DS / 'data.yaml'), imgsz=640, verbose=False).box.map50
print(f'old mAP50={base_map:.3f}  new mAP50={new_map:.3f}')
if new_map > base_map:
    shutil.copy('runs/mvtec_y11n_cont/weights/best.pt', 'runs/mvtec_y11n/weights/best.pt')
    print('improved -> best.pt replaced; re-run sections 5-7')
else:
    print('no improvement -> keeping the original best.pt')


## 5. Doğrulama metrikleri

In [ ]:
import json, yaml
from pathlib import Path

best = Path('runs/mvtec_y11n/weights/best.pt')
m = YOLO(str(best))

# 1. Genel doğrulama seti metrikleri
r = m.val(data=str(DS / 'data.yaml'), imgsz=640, verbose=False)
metrics = {
    'mAP50': float(r.box.map50),
    'mAP50_95': float(r.box.map),
    'per_class_AP50': {CLASSES[int(c)]: float(ap) for c, ap in zip(r.box.ap_class_index, r.box.ap50)},
}

# 2. Uygulama kategorilerine (CATEGORIES) özel doğrulama metrikleri
val_app_file = DS / 'val_app.txt'
if val_app_file.exists() and val_app_file.read_text().strip():
    data_app_yaml = DS / 'data_app.yaml'
    data_app_yaml.write_text(yaml.safe_dump({
        'path': str(DS.resolve()),
        'train': 'images/train',
        'val': 'val_app.txt',
        'names': {i: n for i, n in enumerate(CLASSES)},
    }))
    r_app = m.val(data=str(data_app_yaml), imgsz=640, verbose=False)
    metrics['mAP50_app'] = float(r_app.box.map50)
    data_app_yaml.unlink(missing_ok=True)
else:
    metrics['mAP50_app'] = metrics['mAP50']

OUT = Path('out_yolo')
OUT.mkdir(exist_ok=True)
(OUT / 'yolo_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

## 6. ONNX dışa aktarma ve hızlı test

In [ ]:
onnx_path = m.export(format='onnx', imgsz=640, simplify=True, dynamic=False)
shutil.copy(onnx_path, OUT / 'yolo.onnx')
shutil.copy(best, OUT / 'yolo.pt')
test_model = YOLO(str(OUT / 'yolo.onnx'), task='detect')
for img in sorted((DS / 'images' / 'val').glob('*.png'))[:2]:
    res = test_model.predict(str(img), imgsz=640, conf=0.25, verbose=False)[0]
    print(img.name, [(res.names[int(b.cls)], round(float(b.conf), 2)) for b in res.boxes])

## 7. Çıktıları indir (yolo.onnx + yolo.pt + metrikler)

In [ ]:
zip_path = shutil.make_archive('yolo_out', 'zip', OUT)
print('zip:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not in Colab; file saved at', zip_path)